In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('train.txt', sep=';', header=None, names=['text', 'emotion'])

In [3]:
data.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
data['emotion'].unique()
data['emotion'].value_counts()

emotion
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64

In [5]:
from sklearn.preprocessing import LabelEncoder

label = LabelEncoder()

label.fit(data['emotion'])

data['emotion'] = label.transform(data['emotion'])
data['emotion'].value_counts()

emotion
2    5362
4    4666
0    2159
1    1937
3    1304
5     572
Name: count, dtype: int64

In [6]:
import string

def remove_punc(txt):
    return txt.translate(str.maketrans(" ", " ", string.punctuation))

data['text'] = data['text'].apply(remove_punc)

In [7]:
data['text'] = data['text'].str.lower()
data.head()

,text,emotion
0,i didnt feel humiliated,4
1,i can go from feeling so hopeless to so damned...,4
2,im grabbing a minute to post i feel greedy wrong,0
3,i am ever feeling nostalgic about the fireplac...,3
4,i am feeling grouchy,0


In [8]:
def remove_num(txt):
    new = ""
    for text in txt.split():
        if not text.isdigit():
            new += text + " "
    return new

data['text'] = data['text'].apply(remove_num)
data.head(3)

,text,emotion
0,i didnt feel humiliated,4
1,i can go from feeling so hopeless to so damned...,4
2,im grabbing a minute to post i feel greedy wrong,0


In [9]:
def remove_link(txt):
    new = ""
    for link in txt.split():
        if not link.startswith("hhtp"):
            new += link + " "
    return new

data['text'] = data['text'].apply(remove_link)
data.head(3)

,text,emotion
0,i didnt feel humiliated,4
1,i can go from feeling so hopeless to so damned...,4
2,im grabbing a minute to post i feel greedy wrong,0


In [10]:
def remove_emoji(txt):
    new = ""
    for emoji in txt.split():
        if emoji.isascii():
            new += emoji + " "
    return new

data['text'] = data['text'].apply(remove_emoji)
data.head(3)

,text,emotion
0,i didnt feel humiliated,4
1,i can go from feeling so hopeless to so damned...,4
2,im grabbing a minute to post i feel greedy wrong,0


In [11]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt_tab') # it's a library we are using for tokenization

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\manor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\manor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [12]:
# after import NLTK will do tokenization
stop_words = set(stopwords.words('english'))

def remove_stopwords(txt):
  word_token = word_tokenize(txt)
  new = ""
  for i in word_token:
    if i not in stop_words:
      new += i + " "
  return new

data['text'] = data['text'].apply(remove_stopwords)

In [13]:
data.head()

,text,emotion
0,didnt feel humiliated,4
1,go feeling hopeless damned hopeful around some...,4
2,im grabbing minute post feel greedy wrong,0
3,ever feeling nostalgic fireplace know still pr...,3
4,feeling grouchy,0


In [14]:
X = data['text']
y = data['emotion']

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train , y_test = train_test_split(X, y , test_size=0.2, random_state=42)

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline_lr_tf = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000))
])

pipeline_lr_tf.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()),
                ('classifier', LogisticRegression(max_iter=1000))])

In [17]:
from sklearn.metrics import classification_report
y_pred_tf_lr = pipeline_lr_tf.predict(X_test)

print(classification_report(y_test, y_pred_tf_lr))

              precision    recall  f1-score   support

           0       0.90      0.81      0.86       427
           1       0.86      0.76      0.80       397
           2       0.81      0.96      0.88      1021
           3       0.90      0.61      0.72       296
           4       0.90      0.94      0.92       946
           5       0.88      0.47      0.61       113

    accuracy                           0.86      3200
   macro avg       0.88      0.76      0.80      3200
weighted avg       0.87      0.86      0.86      3200



In [18]:
from sklearn.feature_extraction.text import CountVectorizer

pipeline_lr_bow = Pipeline([
    ('BOW', CountVectorizer()),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline_lr_bow.fit(X_train, y_train)

Pipeline(steps=[('BOW', CountVectorizer()),
                ('classifier', LogisticRegression(max_iter=1000))])

In [19]:
y_pred_lr_bow = pipeline_lr_bow.predict(X_test)

print(classification_report(y_test, y_pred_lr_bow))

              precision    recall  f1-score   support

           0       0.89      0.85      0.87       427
           1       0.86      0.84      0.85       397
           2       0.88      0.94      0.91      1021
           3       0.84      0.76      0.80       296
           4       0.93      0.93      0.93       946
           5       0.86      0.72      0.78       113

    accuracy                           0.89      3200
   macro avg       0.88      0.84      0.86      3200
weighted avg       0.89      0.89      0.89      3200



In [20]:
from sklearn.naive_bayes import MultinomialNB

pipeline_nb_tf = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('naivebayes', MultinomialNB())
])

pipeline_nb_tf.fit(X_train, y_train)


Pipeline(steps=[('tfidf', TfidfVectorizer()), ('naivebayes', MultinomialNB())])

In [21]:
y_pred_nb_tf = pipeline_nb_tf.predict(X_test)

print(classification_report(y_test, y_pred_nb_tf))

              precision    recall  f1-score   support

           0       0.93      0.29      0.44       427
           1       0.92      0.22      0.36       397
           2       0.60      0.99      0.74      1021
           3       1.00      0.03      0.06       296
           4       0.70      0.93      0.80       946
           5       1.00      0.01      0.02       113

    accuracy                           0.66      3200
   macro avg       0.86      0.41      0.40      3200
weighted avg       0.76      0.66      0.58      3200



In [22]:
pipeline_nb_bow = Pipeline([
    ("BOW", CountVectorizer()),
    ("naivebayes", MultinomialNB())
])

pipeline_nb_bow.fit(X_train, y_train)

Pipeline(steps=[('BOW', CountVectorizer()), ('naivebayes', MultinomialNB())])

In [23]:
y_pred_nb_bow = pipeline_nb_bow.predict(X_test)
print(classification_report(y_test, y_pred_nb_bow))

              precision    recall  f1-score   support

           0       0.89      0.63      0.74       427
           1       0.85      0.57      0.68       397
           2       0.73      0.96      0.83      1021
           3       0.93      0.26      0.41       296
           4       0.75      0.95      0.84       946
           5       1.00      0.05      0.10       113

    accuracy                           0.77      3200
   macro avg       0.86      0.57      0.60      3200
weighted avg       0.80      0.77      0.74      3200



In [24]:
import pickle

with open('emotion_pipline.pkl', 'wb') as f:
    pickle.dump(pipeline_lr_bow, f)

In [26]:
pipeline = pickle.load(open("emotion_pipline.pkl", "rb"))

text = "I am very sad today"

prediction = pipeline.predict([text])

print(prediction[0])

4
